# Task 2 — full-data tuned CatBoost candidate (v8)

The v6 hyperparameter winner plus the v7 fixed Social/Whale weights is the highest numerical five-fold development score, though its advantage over the unweighted v6 configuration is negligible. This notebook refits that configuration on **all labeled competition Train rows** after validation. The recorded `macro_F1_cv5_procedure` is the earlier v7 development-fold estimate, **not** a score measured on this full-data fit. All three tier inputs remain temporally unverified. No Test rows are used for fitting or evaluation.

In [1]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
DATA='eda_original_v2'; SOURCE='t2_cat_tuned_weighted_cv5_v7'
NAME='t2_cat_tuned_weighted_full_v8'
data_card=json.loads((SRC/'task2'/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
source_card=json.loads((SRC/'task2'/'save'/f'{SOURCE}.json').read_text(encoding='utf-8'))
assert source_card['data_name']==DATA
train=pd.read_parquet(SRC/data_card['splits']['train']['path'])
y=train.segment.astype(int)
raw=train.drop(columns=['player_id','segment'])
cats=raw.select_dtypes(include=['object']).columns.tolist()
X=restore_categorical_nulls(raw,cats)
assert len(X)==104000 and len(X.columns)==44 and len(cats)==10
source_model=joblib.load(SRC/source_card['model_path'])
pre=source_model.named_steps['preprocess']
params=source_model.named_steps['classifier'].get_params()
assert params['random_seed']==SEED and params['class_weights']==[1.0,1.0,1.25,1.25]
result_path=SRC/'task2'/'models'/'results'/'t2_cat_tuned_weighted_cv5_v7.json'
folds=json.loads(result_path.read_text(encoding='utf-8'))['rows']
assert len(folds)==5
cv_mean=float(np.mean([r['macro_F1'] for r in folds]))
print('Labeled Train rows:',len(X),'development CV procedure F1:',cv_mean)

Labeled Train rows: 104000 development CV procedure F1: 0.7541563024643203


In [2]:
# Fit, save, and record the selected procedure estimate in one cell.
SAVE=SRC/'task2'/'save'
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
if not any(p.exists() for p in paths):
    model=Pipeline([('preprocess',clone(pre)),('classifier',CatBoostClassifier(**params))])
    model.fit(X,y)
    save_model(task=2,name=NAME,model=model,data_name=DATA,
        card={'idea':'Full Train refit of tuned weighted CatBoost candidate',
              'source_model':SOURCE,'rows':len(X),'seed':SEED,
              'parameters':{'depth':params['depth'],'iterations':params['iterations'],
                            'learning_rate':params['learning_rate'],'l2_leaf_reg':params['l2_leaf_reg'],
                            'random_strength':params['random_strength'],
                            'bagging_temperature':params['bagging_temperature'],
                            'class_weights':params['class_weights']},
              'tier_timing':'unverified'})
    evaluation=evaluate_model(task=2,name=NAME,score=cv_mean,
        metric_name='macro_F1_cv5_procedure',higher_is_better=True,
        card={'split':'five development folds from v7, before full Train refit',
              'source_model':SOURCE,'not_full_fit_score':True,'seed':SEED})
else:
    assert all(p.exists() for p in paths),'Inspect incomplete artifact'
    saved=json.loads(paths[1].read_text(encoding='utf-8'))
    assert saved['data_name']==DATA and saved['card']['source_model']==SOURCE
    evaluation=json.loads(paths[2].read_text(encoding='utf-8'))
    assert evaluation['metric']=='macro_F1_cv5_procedure'
print('Saved:',NAME,'CV procedure estimate:',evaluation['score'])

Saved: t2_cat_tuned_weighted_full_v8 CV procedure estimate: 0.7541563024643203
